In [ ]:
import geopandas as gpd

# Preparing for GLASGOW GWR 10x10m 

## Glasgow Converting 10m NDVI

In [ ]:
#C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/winter/gg_sum_10m_rast.tif

### reading the NDVI raster as polygons 

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import shapes
import numpy as np
from shapely.geometry import shape

with rasterio.open(r"C:\Users\ibk1\NoiseModelling\Glasgow\noisemodelling_outputs\Round3\sum_2022_groundcover.geojson") as src:
    noise_array = src.read(1)
    transform = src.transform

results = [
    {"type": "Feature",
     "geometry": shape(geom),
     "properties": {"value": value}}
    for geom, value in shapes(noise_array, transform=transform)
    if value is not None
]

ndvi = gpd.GeoDataFrame.from_features(results, crs=src.crs)

ndvi.rename(columns={"value": "G"}, inplace=True)

In [ ]:
import geopandas as gpd

In [ ]:
ndvi = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_ndvi_clipped.shp")

In [ ]:
ndvi.head()

### joining NDVI with landcover data 
took about ten mins

In [ ]:
landcover = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Results\Landcoverc\gg_landcover.shp")
ndvi = gpd.sjoin(ndvi, landcover[["geometry", "lc_class"]], how="left", predicate="intersects")

In [ ]:
gwr_n_lc = ndvi

In [ ]:
gwr_n_lc.head()

In [ ]:
gwr_n_lc["mean_noise"].describe()

In [ ]:
gwr_n_lc

In [ ]:
from rasterstats import zonal_stats
stats = zonal_stats(ndvi, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_10_rast_filled.tif", stats=["mean"], nodata=None)

In [ ]:
ndvi["mean_noise"] = [s["mean"] for s in stats]

In [ ]:
ndvi.head()

In [ ]:
ndvi["mean_noise"].describe()

## Trees

In [ ]:
import geopandas as gpd
import pandas as pd
import glob
import os

# Folder containing the shapefiles
folder_path = r"C:\Users\ibk1\NoiseModelling\Glasgow\env_data\tree_volume\tree_canopy_products\treetop_location"

# Get a list of all shapefiles in the folder
shapefiles = glob.glob(os.path.join(folder_path, "*.shp"))

# Read and process each shapefile
gdfs = [gpd.read_file(shp).set_crs(epsg=27700, allow_override=True) for shp in shapefiles]

# Merge all shapefiles into a single GeoDataFrame
merged_gdf = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True))
clip_trees = gpd.clip(merged_gdf, ggbounds)

In [ ]:
if "index_right" in gwr_n_lc.columns:
    gwr_n_lc = gwr_n_lc.drop(columns=["index_right"])

In [ ]:
# Here is another way 
tree_counts = gpd.sjoin(clip_trees, gwr_n_lc, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
gwr_n_lc["tree_count"] = gwr_n_lc.index.map(tree_summary).fillna(0).astype(int)

In [ ]:
gwr_n_lc.head()

In [ ]:
gwr_n_lc["lc_class"].unique()

In [ ]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype(str)

# Step 2: Filter out rows where lc_class is '0.0' or 'nan'
gwr_n_lc = gwr_n_lc[~gwr_n_lc["lc_class"].isin(["0.0", "nan"])]

In [ ]:
gwr_n_lc["lc_class"].unique()

# Dummy coding 

In [ ]:
gwr_n_lc["lc_class"] = gwr_n_lc["lc_class"].astype("category")

In [ ]:
gwr_n_lc.head()

In [ ]:
import pandas as pd

# Create dummy variables, drop_first=True avoids multicollinearity (reference category)
dummies = pd.get_dummies(gwr_n_lc["lc_class"], prefix="lc", drop_first=True)

# Join them back to the GeoDataFrame
gwr_n_lc_d = gwr_n_lc.join(dummies)

In [ ]:
gwr_n_lc_d["tree_count"].describe()

In [ ]:
gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_10m.shp")

In [ ]:
gwr_n_lc_d.info

## centriods 

In [ ]:
# Create a new GeoDataFrame with centroids
centroids_gwr_n_lc_d = gwr_n_lc_d.copy()
centroids_gwr_n_lc_d["geometry"] = gwr_n_lc_d.geometry.centroid

In [ ]:
centroids_gwr_n_lc_d.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_10m_centriods.shp")

### Creating Urban pct test

In [ ]:
noise_gdf["grid_id"] = noise_gdf.index.astype(str)

In [ ]:
# testing 
urban_codes = [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]
urban = landcover[landcover["code_2018"].astype(int).isin(urban_codes)]

In [ ]:
urban_overlap = gpd.overlay(noise_gdf, urban, how="intersection")
urban_overlap["urban_area"] = urban_overlap.geometry.area

In [ ]:
urban_by_grid = (
    urban_overlap.groupby("grid_id")["urban_area"]
    .sum()
    .reset_index()
    .rename(columns={"urban_area": "urban_area_m2"})
)

In [ ]:
# Add to full grid
noise_gdf["grid_area"] = noise_gdf.geometry.area  # should be 100 m² for 10x10m
noise_gdf = noise_gdf.merge(urban_by_grid, on="grid_id", how="left")

# Fill NaNs (no urban overlap) with 0
noise_gdf["urban_area_m2"] = noise_gdf["urban_area_m2"].fillna(0)

# Calculate proportion
noise_gdf["urban_pct"] = noise_gdf["urban_area_m2"] / noise_gdf["grid_area"]


### joining with tree count data 

In [ ]:
clip_trees.columns

In [ ]:
noise_gdf.columns

In [ ]:
if "index_right" in noise_gdf.columns:
    noise_gdf = noise_gdf.drop(columns=["index_right"])

In [ ]:
tree_counts = gpd.sjoin(merged_gdf, noise_gdf, how="inner", predicate="within")
tree_summary = tree_counts.groupby("index_right").size()

# Assign counts to the grid
noise_gdf["tree_count"] = noise_gdf.index.map(tree_summary).fillna(0).astype(int)

In [ ]:
tree_summary

In [ ]:
noise_gdf.head()

In [ ]:
noise_gdf["code_2018"].unique()

In [ ]:
noise_gdf_cleaned = noise_gdf.dropna(subset=["code_2018"])

In [ ]:
noise_gdf_cleaned.head()

### Reclassifying landcover to urban, green, or forest 

In [ ]:
clipped.dtypes

In [ ]:
# Convert to numeric
clipped["code_2018"] = pd.to_numeric(clipped["code_2018"], errors="coerce")

In [ ]:
clipped["code_2018"].unique()

In [ ]:
import geopandas as gpd
import numpy as np

# Assume your GeoDataFrame is called gdf
def classify_landcover(code):
    if code in [11100, 11210, 11220, 11230, 11240, 11300, 12100, 12210, 12220, 12230, 12300, 12400, 13100, 13300, 14400]:
        return 'urban'  # Urban
    elif code in [14100, 14200, 21000, 22000, 23000, 24000]:
        return 'green'  # Green
    elif code in [25000, 31000, 32000]:
        return 'forest'  # Forest
    elif code == 50000:
        return 'water'  # Water
    else:
        return 'other'  # Or some other default

# Apply classification
clipped["landcover_group"] = clipped["code_2018"].apply(classify_landcover)

In [ ]:
clipped.head()

In [ ]:
test = clipped[clipped["landcover_group"] == "green"]
test

In [ ]:
test["landcover_group"].unique()

#### Dropping the water category, and all others 

In [ ]:
clipped = clipped[~clipped['landcover_group'].isin(['water', 'other'])]

In [ ]:
clipped

#### setting the correct ordering 

In [ ]:
clipped['landcover_group'] = pd.Categorical(
    clipped['landcover_group'],
    categories=['urban', 'green', 'forest'],
    ordered=True
)
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

In [ ]:
clipped.head()

In [ ]:
#noise_gdf = pd.get_dummies(noise_gdf, columns=['landcover_group'], drop_first=True)

## Scaling Features Between 1 and 0

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
clipped[["noise_norm", "tree_count_norm"]] = scaler.fit_transform(clipped[["mean_noise", "tree_count"]])

In [ ]:
clipped.describe()

In [ ]:
clipped["G"].median()

In [ ]:
clipped.crs

In [ ]:
clipped.head()

In [ ]:
clipped.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\GWRInput_1m.shp")

# Filling in no data values 

In [ ]:
import rasterio
import numpy as np

# Load the raster
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sim_1m_clipped.tif") as src:
    noise = src.read(1)
    profile = src.profile
    nodata = src.nodata or np.nan


In [ ]:
# Create a mask of missing values
mask = np.isnan(noise)


In [ ]:
from scipy.ndimage import generic_filter

# Function to compute mean of non-NaNs
def nanmean_filter(values):
    return np.nanmean(values)

# Apply focal mean (kernel size = 7 → 70m across if 10m resolution)
kernel_size = 7  # Adjust depending on your radius
smoothed = generic_filter(noise, nanmean_filter, size=kernel_size, mode='constant', cval=np.nan)

In [ ]:
# Correct mask
mask = (noise == nodata) if nodata is not None else np.isnan(noise)

# Smoothing kernel (e.g., 7x7 ~ 70m window)
def nanmean_filter(values):
    return np.nanmean(values)

smoothed = generic_filter(noise.astype(float), nanmean_filter, size=7, mode='constant', cval=np.nan)

# Fill missing areas
filled = np.where(mask, smoothed, noise)
profile.update(dtype=rasterio.float32, nodata=None)

In [ ]:
filled = np.where(mask, smoothed, noise)

In [ ]:
with rasterio.open("C:/Users/ibk1/NoiseModelling/Glasgow/Analysis/summer/gg_sum_1m_filled.tif", "w", **profile) as dst:
    dst.write(filled, 1)

# Vegetation GWR for 50 by 50m Grid

In [ ]:
import geopandas as gpd 
ndns = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\50x50_ndvi_noise_grid.geojson")

In [ ]:
ndns.head()

In [ ]:
from rasterstats import zonal_stats
stats = zonal_stats(ndns, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\cluster analysis\lc_cls_v2_50by50grid.tif", categorical=True)

In [ ]:
import pandas as pd 

In [ ]:
stats_df = pd.DataFrame(stats).fillna(0)  # fill NaN with 0 where class is missing
stats_df.columns = [f'class_{int(col)}' for col in stats_df.columns]

ndns = ndns.reset_index(drop=True)  # Ensure same index
ndns = pd.concat([ndns, stats_df], axis=1)
# Total pixels (can also use sum of class columns)
ndns['total_pix'] = stats_df.sum(axis=1)

# Example: forest (class 3) percentage
ndns['forest_pct'] = ndns['class_3'] / ndns['total_pix']
ndns['grass_pct'] = ndns['class_2'] / ndns['total_pix']
ndns['urban_pct'] = ndns['class_1'] / ndns['total_pix']


In [ ]:
ndns.head()

In [ ]:
ndns.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\cluster analysis\cluster_input_gg.gpkg", driver="GPKG")

In [ ]:
stats = zonal_stats(ndns, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\lc_class_rast_gg.tif", categorical=True)

# Each dict in `stats` will have keys like: {1: count, 2: count, 3: count}
# Add them to your GeoDataFrame
ndns["urban_count"] = [s.get(1, 0) for s in stats]
ndns["green_count"] = [s.get(2, 0) for s in stats]
ndns["forest_count"] = [s.get(3, 0) for s in stats]

In [ ]:
ndns["total"] = ndns[["urban_count", "green_count", "forest_count"]].sum(axis=1)
ndns["urban_share"] = ndns["urban_count"] / ndns["total"]

In [ ]:
ndns.head()

In [ ]:
from geopandas.tools import sjoin

# Perform the spatial join only once, with how="left"
joined = sjoin(ndns, clip_trees, how="left")

# Now group by the index of the left GeoDataFrame (ndns)
tree_counts = joined.groupby(joined.index).size()

# Add counts to the original GeoDataFrame
ndns["tree_count"] = ndns.index.map(tree_counts).fillna(0).astype(int)


In [ ]:
ndns.head()

In [ ]:
ggbounds.explore()

In [ ]:
ndns["x"] = ndns.geometry.centroid.x
ndns["y"] = ndns.geometry.centroid.y

In [ ]:
ndns.head()

In [ ]:
ndns[["geometry","x", "y", "noise_mean", "ndvi_mean", "tree_count", "urban_count", "green_count", "forest_count"]].to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_gwr_input_50m.shp", index=False)

In [ ]:
len(ndns)

# GWR Preprocessing by Datazones

In [ ]:
# Reading the datazones shapefile 
ggdz = gpd.read_file(r"C:/Users/ibk1/NoiseModelling/Glasgow/env_data/gg_te/gg_te.shp")
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_1m_filled.tif", categorical=False)
ggdz["mean_noise"] = [s["mean"] for s in stats]

In [ ]:
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_sum_1m_filled.tif", categorical=False)

In [ ]:
ggdz["mean_noise"] = [s["mean"] for s in stats]

In [ ]:
ggdz.columns

In [ ]:
ggdz = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_GWR_Input_summer_datazones.shp") 

In [ ]:
ggdz.explore(column = "apb_index")

In [ ]:
ggdz.explore(column = "apb_index")

In [ ]:
SIMD = gpd.read_file(r"C:\Users\ibk1\NoiseModelling\SIMD Data\SG_SIMD_2020.shp")

In [ ]:
SIMD.head()

In [ ]:
SIMD.columns

In [ ]:
ggsimd = SIMD[SIMD["LAName"] == "Glasgow City"]

In [ ]:
ggdz.head(1)

In [ ]:
ggsimd.columns

In [ ]:
cols_to_keep = ['DataZone','HlthCIF',
       'HlthAlcSR', 'HlthDrugSR', 'HlthSMR', 'HlthDprsPc', 'HlthLBWTPc',
       'HlthEmergS', 'HlthRank','HouseNumOC', 'HouseNumNC', 'HouseOCrat',
       'HouseNCrat', 'HouseRank', "Vigintilv2",'EduRank', "GAccRank", "CrimeRate"
               ]
merged = ggdz.merge(ggsimd[cols_to_keep], left_on="bge_code", right_on="DataZone", how="left")


In [ ]:
merged.explore(column = "HlthDprsPc")

In [ ]:
# Remove % and convert to float
merged["HouseOCrat"] = merged["HouseOCrat"].str.replace("%", "").astype(float)
merged["HouseNCrat"] = merged["HouseNCrat"].str.replace("%", "").astype(float)
merged["HlthDprsPc"] = merged["HlthDprsPc"].str.replace("%", "").astype(float)
merged["HlthLBWTPc"] = merged["HlthLBWTPc"].str.replace("%", "").astype(float)
# Optional: Convert to decimals (0.1 instead of 10)
# ggsimd["HouseOCrat"] /= 100
# ggsimd["HouseNCrat"] /= 100

In [ ]:
merged.columns

In [ ]:
merged.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\summer\gg_GWR_Input_summer_datazones_SIMD_extra.shp") 

## Winter 

In [ ]:
# Reading the datazones shapefile 
ggdz_wint = gpd.read_file(r"C:/Users/ibk1/NoiseModelling/Glasgow/env_data/gg_te/gg_te.shp")
stats = zonal_stats(ggdz, r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\winter\gg_winter_1m_raster.tif", categorical=False)
ggdz_wint["mean_noise"] = [s["mean"] for s in stats]

In [ ]:
ggdz_wint.to_file(r"C:\Users\ibk1\NoiseModelling\Glasgow\Analysis\winter\gg_GWR_Input_WINTER_datazones.shp") 